# `02` — Mixed Activation Functions on California Housing

This experiment investigates whether **heterogeneous activation functions** within a single layer — the core idea behind `MultiDense` — can outperform fully homogeneous architectures.

The question being asked is:

> *Does mixing ReLU and TanH neurons within the same layer, in varying proportions, yield better regression performance than using a single activation function throughout?*

To answer it, seven models are trained and evaluated on the [California Housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) — a real-world regression benchmark with 20,640 samples and 8 features, where the target is the median house value (in hundreds of thousands of dollars) for California districts. It is the standard replacement for the deprecated Boston Housing dataset.

**Experimental models (EM)** use `MultiDense` with varying ReLU/TanH split ratios.

**Reference models (RM)** use native `Dense` with a single activation, providing the homogeneous baselines that the mixed models are measured against.

----

## 1. Imports

scikit-learn is used exclusively for loading and splitting the dataset. All model definitions and training are handled by TensorFlow/Keras.

In [16]:
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

from multi_dense import MultiDense

----

## 2. Reproducibility

A global random seed is set before anything else so that weight initialisation, data shuffling, and dropout (if added later) are all deterministic. The seed is reset again immediately before each model's `fit()` call, so that every model starts from the same RNG state regardless of how many models have been built or evaluated before it.

In [17]:
SEED = 42

In [18]:
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

----

## 3. Model Definitions

All models share the same architecture shape — three hidden layers of 64, 64, and 32 neurons, followed by a single linear output neuron — so that the only variable across models is the activation function assignment.

Models are divided into two groups:

**Experimental models (EM)** — built with `MultiDense`, sweeping the ReLU/TanH split ratio from 100/0 to 0/100 in 25% steps:

| Tag | ReLU neurons | TanH neurons | Layer sizes |
|-----|-------------|--------------|-------------|
| `EM:100+0` | 100% | 0% | [64], [64], [32] |
| `EM:75+25` | 75% | 25% | [48,16], [48,16], [24,8] |
| `EM:50+50` | 50% | 50% | [32,32], [32,32], [16,16] |
| `EM:25+75` | 25% | 75% | [16,48], [16,48], [8,24] |
| `EM:0+100` | 0% | 100% | [64], [64], [32] |

**Reference models (RM)** — built with native `Dense`, providing the homogeneous baselines:

| Tag | Activation | Description |
|-----|-----------|-------------|
| `RM:100+0` | ReLU | Pure ReLU baseline |
| `RM:0+100` | TanH | Pure TanH baseline |

> `EM:100+0` and `EM:0+100` are mathematically identical to `RM:100+0` and `RM:0+100` respectively. Their inclusion verifies that `MultiDense` with a single partition produces the same results as native `Dense`.

In [19]:
models: dict[str, Sequential] = dict()

### 3.1. Experimental models — `MultiDense`

Each hidden layer is expressed as a `MultiDense` whose partitions sum to the same total neuron count as the reference models. The ReLU partition always comes first, followed by TanH.

In [20]:
# EM:100+0 — pure ReLU via MultiDense (single partition)
models["EM:100+0"] = Sequential(
    [
        MultiDense([64], activations=["relu"]),
        MultiDense([64], activations=["relu"]),
        MultiDense([32], activations=["relu"]),
        Dense(1, activation="linear"),
    ]
)

In [21]:
# EM:75+25 — 75% ReLU, 25% TanH
models["EM:75+25"] = Sequential(
    [
        MultiDense([48, 16], activations=["relu", "tanh"]),
        MultiDense([48, 16], activations=["relu", "tanh"]),
        MultiDense([24, 8], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

In [22]:
# EM:50+50 — equal split
models["EM:50+50"] = Sequential(
    [
        MultiDense([32, 32], activations=["relu", "tanh"]),
        MultiDense([32, 32], activations=["relu", "tanh"]),
        MultiDense([16, 16], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

In [23]:
# EM:25+75 — 25% ReLU, 75% TanH
models["EM:25+75"] = Sequential(
    [
        MultiDense([16, 48], activations=["relu", "tanh"]),
        MultiDense([16, 48], activations=["relu", "tanh"]),
        MultiDense([8, 24], activations=["relu", "tanh"]),
        Dense(1, activation="linear"),
    ]
)

In [24]:
# EM:0+100 — pure TanH via MultiDense (single partition)
models["EM:0+100"] = Sequential(
    [
        MultiDense([64], activations=["tanh"]),
        MultiDense([64], activations=["tanh"]),
        MultiDense([32], activations=["tanh"]),
        Dense(1, activation="linear"),
    ]
)

### 3.2. Reference models — native `Dense`

Identical architecture, but using the built-in Keras `Dense` layer with a single activation. These are the gold-standard homogeneous baselines.

In [25]:
# RM:100+0 — pure ReLU via Dense
models["RM:100+0"] = Sequential(
    [
        Dense(64, activation="relu"),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1, activation="linear"),
    ]
)

In [26]:
# RM:0+100 — pure TanH via Dense
models["RM:0+100"] = Sequential(
    [
        Dense(64, activation="tanh"),
        Dense(64, activation="tanh"),
        Dense(32, activation="tanh"),
        Dense(1, activation="linear"),
    ]
)

### 3.3. Compilation

All models are compiled with MSE loss and MAE as the reported metric. MAE is more interpretable here because it is in the same unit as the target (hundreds of thousands of dollars), making it straightforward to compare prediction errors across models.

In [27]:
for tag, model in models.items():
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"],
    )
    print(f"{tag} compiled")

EM:100+0 compiled
EM:75+25 compiled
EM:50+50 compiled
EM:25+75 compiled
EM:0+100 compiled
RM:100+0 compiled
RM:0+100 compiled


----

## 4. Dataset — California Housing

The California Housing dataset contains one row per census district and was derived from the 1990 U.S. Census. Each of the 20,640 samples describes a district through 8 features:

| Feature | Description |
|---------|-------------|
| `MedInc` | Median income (tens of thousands of USD) |
| `HouseAge` | Median house age in the district |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | District population |
| `AveOccup` | Average household occupancy |
| `Latitude` | District latitude |
| `Longitude` | District longitude |

The target is the **median house value** for each district, expressed in hundreds of thousands of dollars (e.g. a target of `2.5` means $250,000).

Compared to the Boston Housing dataset used in the previous experiment, California Housing offers roughly 40× more samples (20,640 vs 506), making the results significantly more statistically stable.

### 4.1. Load

In [28]:
data = fetch_california_housing()

x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(
    data.data.astype("float32"),
    data.target.astype("float32"),
    test_size=0.2,
    random_state=SEED,
)

print(f"Train: {x_train_raw.shape}, Test: {x_test_raw.shape}")
print(
    f"Target range: [{y_train_raw.min():.2f}, {y_train_raw.max():.2f}] "
    f"(mean: {y_train_raw.mean():.2f})"
)

Train: (16512, 8), Test: (4128, 8)
Target range: [0.15, 5.00] (mean: 2.07)


### 4.2. Preprocessing

Both features and target are standardised using training-set statistics only, to avoid data leakage from the test set. Standardising the target is important for regression with MSE loss: it keeps the gradient scale consistent across experiments and makes the loss values directly comparable between models.

The target statistics are saved so that MAE can be de-normalised back to real dollar values in the conclusions.

In [29]:
# Standardise features (fit on train only)
x_mean = x_train_raw.mean(axis=0)
x_std = x_train_raw.std(axis=0)

x_train = (x_train_raw - x_mean) / x_std
x_test = (x_test_raw - x_mean) / x_std

# Standardise target (fit on train only)
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()

y_train = (y_train_raw - y_mean) / y_std
y_test = (y_test_raw - y_mean) / y_std

print(f"x_train: mean≈0 → {x_train.mean(axis=0).round(3)}")
print(f"y_train: mean={y_train.mean():.4f}, std={y_train.std():.4f}")

x_train: mean≈0 → [ 0.  0.  0. -0.  0.  0. -0. -0.]
y_train: mean=-0.0000, std=1.0000


----

## 5. Training

Each model is trained independently. Crucially, the global random seed is reset to the same value before every `fit()` call. This ensures that all models start from identical RNG states, so any difference in results reflects the activation configuration rather than lucky or unlucky weight initialisation.

Early stopping is used to prevent overfitting and to avoid manually tuning the number of epochs per model. It monitors validation loss and restores the weights from the best epoch before returning.

In [30]:

histories: dict[str, tf.keras.callbacks.History] = {}

for tag, model in models.items():
    tf.keras.utils.set_random_seed(SEED)  # reset before each model
    print(f"Training {tag}...")

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
    )

    histories[tag] = model.fit(
        x_train,
        y_train,
        epochs=300,
        batch_size=256,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0,
    )

    stopped_at = len(histories[tag].history["loss"])
    print(f"  → stopped at epoch {stopped_at}\n")

Training EM:100+0...


TypeError: Expected any non-tensor type, but got a tensor instead.

----

## 6. Evaluation

Models are evaluated on the held-out test set. MAE is de-normalised by multiplying by `y_std` to recover the error in the original unit (hundreds of thousands of dollars), then converted to dollars for readability.

In [ ]:
results: dict[str, dict] = {}

for tag, model in models.items():
    loss, mae = model.evaluate(x_test, y_test, verbose=0)
    mae_dollars = mae * y_std * 100_000
    results[tag] = {"loss": loss, "mae": mae, "mae_dollars": mae_dollars}

----

## 7. Conclusions

Results are sorted by MAE (ascending) so the best-performing model appears first. The `▲` and `▼` markers indicate whether a model beat or lost to the better of the two homogeneous `Dense` baselines (`RM:100+0` and `RM:0+100`).

In [ ]:
# Best MAE among the two pure-Dense reference models
best_rm_mae = min(results["RM:100+0"]["mae"], results["RM:0+100"]["mae"])

print(f"{'Model':<14} {'MAE (norm)':>10} {'MAE ($)':>12} {'vs best RM':>12}")
print("-" * 52)

for tag, r in sorted(results.items(), key=lambda x: x[1]["mae"]):
    delta = r["mae"] - best_rm_mae
    marker = "▲ better" if delta < -1e-4 else ("▼ worse" if delta > 1e-4 else "≈ same")
    print(f"{tag:<14} {r['mae']:>10.4f} {r['mae_dollars']:>11,.0f}€  {marker}")

Model          MAE (norm)      MAE ($)   vs best RM
----------------------------------------------------
EM:100+0           0.3064      35,426€  ▲ better
EM:50+50           0.4174      48,260€  ▲ better
EM:25+75           0.4201      48,570€  ▲ better
EM:75+25           0.4242      49,044€  ▲ better
RM:100+0           0.4385      50,700€  ≈ same
RM:0+100           0.4397      50,843€  ▼ worse
EM:0+100           0.4416      51,054€  ▼ worse


----